In [0]:
# ============================================
# 02 - TRANSFORM F1 DATA
# Ingested ADLS → Presentation ADLS
# ============================================

from pyspark.sql.functions import explode, col

ingested_path = "abfss://azure-data@azuredataengine.dfs.core.windows.net/Ingested/"
presentation_path = "abfss://azure-data@azuredataengine.dfs.core.windows.net/Presentation/"

# Read data from the Ingested layer
df_ingested = (
    spark.read
    .option("multiline", "true")
    .json(ingested_path)
)

display(df_ingested)

In [0]:
# Flatten the nested Races array

df_races = df_ingested.select(
    explode(col("MRData.RaceTable.Races")).alias("race")
)

display(df_races)

In [0]:
# Flatten race and circuit information

df_races_flat = df_races.select(
    col("race.season").alias("season"),
    col("race.round").alias("round"),
    col("race.raceName").alias("race_name"),
    col("race.date").alias("race_date"),
    col("race.Circuit.circuitId").alias("circuit_id"),
    col("race.Circuit.circuitName").alias("circuit_name"),
    col("race.Circuit.Location.locality").alias("locality"),
    col("race.Circuit.Location.country").alias("country"),
    col("race.Circuit.Location.lat").alias("latitude"),
    col("race.Circuit.Location.long").alias("longitude")
)

display(df_races_flat)

In [0]:
# Remove duplicate race records

df_clean = df_races_flat.dropDuplicates([
    "season",
    "round",
    "race_name"
])

display(df_clean)

In [0]:
# Write cleaned data to the Presentation layer

df_clean.write \
    .mode("overwrite") \
    .parquet(presentation_path)

print("Clean F1 data successfully written to Presentation layer.")

In [0]:
# Verify the Presentation layer

df_presentation = spark.read.parquet(presentation_path)

display(df_presentation)

In [0]:
df_clean.write \
    .mode("overwrite") \
    .parquet(presentation_path)